In [1]:
import pandas as pd
import numpy as np

# 1. Load the comprehensive 10-year hospital dataset
url = "https://raw.githubusercontent.com/andrewwlong/diabetes_readmission/master/diabetic_data.csv"
df_diabetes = pd.read_csv(url)

print("--- Clinical Data Structural Audit ---")
print(f"Total Patient Encounters (Rows): {df_diabetes.shape[0]}")
print(f"Total Clinical Features (Columns): {df_diabetes.shape[1]}")

--- Clinical Data Structural Audit ---
Total Patient Encounters (Rows): 101766
Total Clinical Features (Columns): 50


In [2]:
# 1. Replace the raw string '?' with real NumPy NaN values
df_diabetes.replace('?', np.nan, inplace=True)

# 2. Calculate the exact percentage of missing data for every single column
missing_percentages = (df_diabetes.isnull().sum() / len(df_diabetes)) * 100
# 3. Filter and display only the columns that actually have missing values
print("--- Real Missing Data Percentages ---")
print(missing_percentages[missing_percentages > 0].sort_values(ascending=False))

--- Real Missing Data Percentages ---
weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64


In [3]:
# 1. Drop columns by explicitly re-assigning the dataframe (No inplace=True)
df_diabetes = df_diabetes.drop(['weight', 'payer_code'], axis=1)

# 2. THE FIX: Explicitly overwrite the column with the filled version
df_diabetes['medical_specialty'] = df_diabetes['medical_specialty'].fillna('Unknown')

# 3. Drop the remaining tiny fractions of missing rows cleanly
df_diabetes = df_diabetes.dropna(subset=['race', 'gender', 'diag_1', 'diag_2', 'diag_3'])

print("--- Phase 1 Cleaning Complete (Warning Free!) ---")
print(f"Remaining Missing Values: {df_diabetes.isnull().sum().sum()}")
print(f"Current Dataset Shape: {df_diabetes.shape}")

--- Phase 1 Cleaning Complete (Warning Free!) ---
Remaining Missing Values: 174705
Current Dataset Shape: (98053, 48)


In [4]:
# 4. Sort by encounter ID to keep things chronological, then drop duplicate patients
df_diabetes.sort_values('encounter_id', inplace=True)
df_diabetes.drop_duplicates(subset=['patient_nbr'], keep='first', inplace=True)

# 5. Drop the ID columns since they are just random database trackers
df_diabetes.drop(['encounter_id', 'patient_nbr'], axis=1, inplace=True)

print("\n--- Identity Deduplication Complete ---")
print(f"Pruned Dataset Shape (Unique Patients Only): {df_diabetes.shape}")


--- Identity Deduplication Complete ---
Pruned Dataset Shape (Unique Patients Only): (68630, 46)


In [5]:
# Check the unique value distributions for these two clinical tests
print("--- Max Glucose Serum Test Distribution ---")
print(df_diabetes['max_glu_serum'].value_counts())

print("\n--- A1C Test Result Distribution ---")
print(df_diabetes['A1Cresult'].value_counts())

--- Max Glucose Serum Test Distribution ---
max_glu_serum
Norm    1686
>200     939
>300     734
Name: count, dtype: int64

--- A1C Test Result Distribution ---
A1Cresult
>8      5793
Norm    3683
>7      2805
Name: count, dtype: int64


In [6]:
# 390–459 or 785: Circulatory System (Heart disease, strokes)

# 460–519 or 786: Respiratory System (Pneumonia, asthma)

# 520–579 or 787: Digestive System (Ulcers, appendicitis)

# 250.xx: Diabetes Mellitus

# 800–999: Injury and Poisoning

# 710–739: Musculoskeletal System

# 580–629 or 788: Genitourinary System

# 140–239: Neoplasms (Cancers/Tumors)

# Everything else: Other categories

def map_icd9_to_category(code):
    """Maps a raw clinical ICD-9 code string into 1 of 9 broad disease categories."""
    if pd.isnull(code):
        return 'Unknown'
    
    code_str = str(code).strip()
    
    # Handle codes that start with letters (E or V codes represent external injuries/supplements)
    if code_str.startswith(('E', 'V')):
        return 'Injury'
        
    # Extract the numeric part before any decimal point to check ranges
    try:
        val = float(code_str.split('.')[0])
    except ValueError:
        return 'Other'
        
    # Apply official clinical range thresholds
    if 390 <= val <= 459 or val == 785:
        return 'Circulatory'
    elif 460 <= val <= 519 or val == 786:
        return 'Respiratory'
    elif 520 <= val <= 579 or val == 787:
        return 'Digestive'
    elif val == 250:
        return 'Diabetes'
    elif 800 <= val <= 999:
        return 'Injury'
    elif 710 <= val <= 739:
        return 'Musculoskeletal'
    elif 580 <= val <= 629 or val == 788:
        return 'Genitourinary'
    elif 140 <= val <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

# Apply the custom clinical mapping function to all 3 diagnosis columns
for col in ['diag_1', 'diag_2', 'diag_3']:
    df_diabetes[col] = df_diabetes[col].apply(map_icd9_to_category)

print("--- ICD-9 High-Cardinality Grouping Complete ---")
print("\nNew unique values for Primary Diagnosis (diag_1):")
print(df_diabetes['diag_1'].value_counts())

--- ICD-9 High-Cardinality Grouping Complete ---

New unique values for Primary Diagnosis (diag_1):
diag_1
Circulatory        21256
Other              10991
Respiratory         9454
Digestive           6337
Injury              5464
Diabetes            5188
Musculoskeletal     3888
Genitourinary       3415
Neoplasms           2637
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Binarize the target variable based on the 30-day clinical penalty mark
df_diabetes['target'] = df_diabetes['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

# 2. Select high-signal features for modeling and drop unneeded tracker columns
features_to_keep = [
    'race', 'gender', 'age', 'time_in_hospital', 'medical_specialty',
    'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient',
    'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed'
]

X_raw = df_diabetes[features_to_keep]
y = df_diabetes['target']

# 3. Convert all categorical text features into pure math columns (0 or 1)
X_encoded = pd.get_dummies(X_raw, drop_first=True, dtype=int)

# 4. Perform our clean Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

# 5. Scale our features uniformly using a fresh StandardScaler instance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train our Random Forest Classifier balanced for class imbalances
print("Training the clinical prediction engine... (This may take 20-30 seconds)")
model_diabetes = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1,class_weight="balanced")
model_diabetes.fit(X_train_scaled, y_train)

# 7. Generate predictions and evaluate the performance matrix
y_pred = model_diabetes.predict(X_test_scaled)
y_prob = model_diabetes.predict_proba(X_test_scaled)[:, 1]

print("\n--- Clinical Classification Performance Report ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

Training the clinical prediction engine... (This may take 20-30 seconds)

--- Clinical Classification Performance Report ---
              precision    recall  f1-score   support

           0       0.93      0.79      0.85     12501
           1       0.14      0.35      0.20      1225

    accuracy                           0.75     13726
   macro avg       0.53      0.57      0.53     13726
weighted avg       0.85      0.75      0.79     13726

ROC-AUC Score: 0.6153


In [8]:
import joblib

# 1. Save the trained model to a file
joblib.dump(model_diabetes, 'diabetes_model.pkl')

# 2. Save the scaler (so the web app can scale new data exactly the same way)
joblib.dump(scaler, 'scaler.pkl')

print("Model and Scaler successfully saved to your hard drive!")

Model and Scaler successfully saved to your hard drive!
